# HAIR CAPSTONE GPU demo
Select a Kaggle T4 GPU, enable Internet, attach the private Kaggle Dataset containing `adapter.safetensors` and `metadata.json`, and enable the `HAIRCAPSTONE_API_KEY` Kaggle Secret for this notebook. Run the next cell. It prints READY only after the model, adapter, local API and public health check pass. Keep the Kaggle session alive while using the local app.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import shutil, subprocess, sys

repo = Path('/kaggle/working/CometicsAI')
url = 'https://github.com/mark-juswa/CometicsAI.git'
inputs = Path('/kaggle/input')
code_dirs = sorted({p.parent.parent for p in inputs.rglob('kaggle_inference_bootstrap.py') if (p.parent.parent / 'backend/app/styles.py').is_file()})
code_zips = sorted(inputs.rglob('haircapstone_runtime_code.zip'))
if len(code_dirs) > 1 or (not code_dirs and len(code_zips) > 1):
    raise RuntimeError('Attach only one HAIR CAPSTONE source-code Input')
if code_dirs:
    shutil.copytree(code_dirs[0], repo, dirs_exist_ok=True)
elif code_zips:
    repo.mkdir(parents=True, exist_ok=True)
    with ZipFile(code_zips[0]) as source:
        if any(not (repo / name).resolve().is_relative_to(repo.resolve()) for name in source.namelist()):
            raise RuntimeError('Unsafe path in code archive')
        source.extractall(repo)
elif (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
elif repo.exists() and any(repo.iterdir()):
    raise RuntimeError(f'{repo} exists but is not the project checkout')
else:
    subprocess.run(['git', 'clone', '--depth', '1', url, str(repo)], check=True)
if not (repo / 'scripts/kaggle_inference_bootstrap.py').is_file():
    raise RuntimeError('Project source lacks the current inference bootstrap; attach the code ZIP Input')
subprocess.run([sys.executable, '-u', str(repo / 'scripts/kaggle_inference_bootstrap.py')], check=True)
